# SCE to adata object conversion for analysis in python

### Load packages

In [1]:
import anndata2ri
import scanpy as sc
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'anndata2ri'

In [ ]:
def matrix(df):
  temp = pd.DataFrame.sparse.from_spmatrix(df)
  return temp

### Haem

In [ ]:
anndata2ri.activate()

In [ ]:
%reload_ext rpy2.ipython

In [ ]:
%%R -o adata
adata <- readRDS("/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/haem/big_sce_norm.rds")

In [ ]:
adata.X = adata.layers['logcounts']

In [ ]:
### add metadata
metadata = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/haem/big_meta_mapping.tab"
meta = pd.read_csv(metadata, '\t')

meta = meta.set_index('cell')
meta['sizeFactor'] = adata.obs.loc[:, 'sizeFactor']
adata.obs = meta

# genes 
genes_in = '/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/data/big_genes.tab'
genes = pd.read_csv(genes_in, '\t')

# PCs
corrected_pcs = '/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/haem/corrected_pc_complete.csv'
pcs = pd.read_csv(corrected_pcs, ',', index_col=0)

adata.obsm['X_pca'] = pcs.to_numpy()

In [ ]:
adata.write("/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/haem/mixl_chim_atlas_haem.h5")

In [ ]:
adata = sc.read("/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/haem/mixl_chim_atlas_haem.h5")

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)

In [ ]:
sc.tl.umap(adata)
sc.pl.umap(adata, color='haem_subclust')

In [ ]:
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')
adata = adata[adata.obs['origin'] == 'chimaera']

In [ ]:
sc.tl.umap(adata)
sc.pl.umap(adata, color='haem_subclust.mapped')

In [ ]:
sc.tl.paga(adata, groups='haem_subclust.mapped')

In [ ]:
sc.pl.paga(adata, color='tdTom', threshold = 0, edge_width_scale = 0.3, fontsize = 10)